In [1]:
import pandas as pd
import re

ausgrid = pd.read_csv("Ausgrid_solar_home_data/Solar home 2012-2013.csv", skiprows=1, parse_dates=["date"])
ausgrid = ausgrid[ausgrid["Customer"].isin([1,2,3,4,5])]
ausgrid.head()

,Customer,Generator Capacity,Postcode,Consumption Category,date,0:30,1:00,1:30,2:00,2:30,...,20:00,20:30,21:00,21:30,22:00,22:30,23:00,23:30,0:00,Row Quality
0,1,3.78,2076,CL,1/07/2012,1.250,1.250,1.250,1.263,0.131,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.081,NaN
1,1,3.78,2076,GC,1/07/2012,0.855,0.786,0.604,0.544,0.597,...,0.374,0.447,0.549,0.136,0.288,0.181,0.651,0.090,0.068,NaN
2,1,3.78,2076,GG,1/07/2012,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,NaN
3,1,3.78,2076,CL,2/07/2012,1.250,1.250,1.125,0.000,0.925,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.069,NaN
4,1,3.78,2076,GC,2/07/2012,0.309,0.082,0.059,0.097,0.290,...,0.353,0.464,0.229,0.811,0.222,0.306,1.034,0.136,0.067,NaN


In [2]:
hourly_regex = re.compile(r"^\d{1,2}:\d{2}$")
hourly_columns = [col for col in ausgrid.columns if hourly_regex.match(col)]
ausgrid = ausgrid.melt(id_vars=["Customer", "Consumption Category", "date"], 
                  value_vars=hourly_columns,
                    var_name="hourly", value_name="kwh"
                  )
print(ausgrid.shape)
print(ausgrid['hourly'].nunique())
print(ausgrid['Consumption Category'].unique())

(251136, 5)
48
<ArrowStringArray>
['CL', 'GC', 'GG']
Length: 3, dtype: str


* General Comsuption (GC) -> load_kwh
* Gross Generation (GG) -> pv_kwh
* Controlled Load (CL) -> Agent decision

In [3]:
ausgrid_wide = ausgrid.pivot_table(
    index=["Customer", "date", "hourly"],
    columns="Consumption Category",
    values="kwh",
    aggfunc="sum",
).reset_index()
ausgrid_wide = ausgrid_wide.rename(columns={"GC": "load_kwh", "GG": "pv_kwh",}).drop(columns=["CL"])   
ausgrid_wide.columns.name = None
print(ausgrid_wide.head())
print(ausgrid_wide.isna().sum())

   Customer       date hourly  load_kwh  pv_kwh
0         1  1/01/2013   0:00     0.097   0.000
1         1  1/01/2013   0:30     0.287   0.000
2         1  1/01/2013  10:00     0.156   1.081
3         1  1/01/2013  10:30     0.142   1.238
4         1  1/01/2013  11:00     0.158   1.338
Customer    0
date        0
hourly      0
load_kwh    0
pv_kwh      0
dtype: int64


In [ ]:
ausgrid_wide['timestamp'] = pd.to_datetime(
    ausgrid_wide['date'] + ' ' + ausgrid_wide['hourly'],
    dayfirst=True
)
is_midnight = ausgrid_wide['hourly'] == '0:00'
ausgrid_wide.loc[is_midnight, 'timestamp'] += pd.Timedelta(days=1)

ausgrid_wide['timestamp'] = (
    ausgrid_wide['timestamp']
    .dt.tz_localize('Australia/Sydney', ambiguous='NaT', nonexistent='NaT')
    .dt.tz_convert('UTC')
)
n_nat = ausgrid_wide['timestamp'].isna().sum()
ausgrid_wide = ausgrid_wide.dropna(subset=['timestamp'])

linhas NaT (viradas de DST) descartadas: 20


In [5]:
day_one = ausgrid_wide[(ausgrid_wide['Customer']==1) & (ausgrid_wide['date']=='1/07/2012')]
print(day_one.sort_values('timestamp')[['hourly','timestamp']].iloc[[0, -1]])

    hourly                 timestamp
289   0:30 2012-06-30 14:30:00+00:00
288   0:00 2012-07-01 14:00:00+00:00


In [ ]:
import glob
prices = pd.concat([pd.read_csv(f) for f in glob.glob("aemo/*.csv")], ignore_index=True)
prices['timestamp'] = (
    pd.to_datetime(prices['SETTLEMENTDATE'])
    .dt.tz_localize('Etc/GMT-10')
    .dt.tz_convert('UTC')
)
prices.rename(columns={'TOTALDEMAND': 'total_demand_mw'}, inplace=True)
prices['price_per_kwh'] = prices['RRP'] / 1000
print(prices.head())
print(prices.shape)
print(prices['price_per_kwh'].describe())

  REGION       SETTLEMENTDATE  total_demand_mw    RRP PERIODTYPE  \
0   NSW1  2013/02/01 00:30:00          7472.60  50.31      TRADE   
1   NSW1  2013/02/01 01:00:00          7193.75  47.91      TRADE   
2   NSW1  2013/02/01 01:30:00          6876.70  49.49      TRADE   
3   NSW1  2013/02/01 02:00:00          6650.65  46.54      TRADE   
4   NSW1  2013/02/01 02:30:00          6527.68  45.73      TRADE   

                  timestamp  price_per_kwh  
0 2013-01-31 14:30:00+00:00        0.05031  
1 2013-01-31 15:00:00+00:00        0.04791  
2 2013-01-31 15:30:00+00:00        0.04949  
3 2013-01-31 16:00:00+00:00        0.04654  
4 2013-01-31 16:30:00+00:00        0.04573  
(17520, 7)
count    17520.000000
mean         0.055102
std          0.012929
min         -0.059280
25%          0.050070
50%          0.052800
75%          0.056050
max          0.317970
Name: price_per_kwh, dtype: float64


In [7]:
ausgrid_merge = pd.merge(
    ausgrid_wide,
    prices[['timestamp', 'price_per_kwh', 'total_demand_mw']],
    on='timestamp',
    how='inner'
)
print('Before:', len(ausgrid_wide))
print('After:', len(ausgrid_merge))
print(ausgrid_merge[['price_per_kwh','total_demand_mw']].isna().sum())

Before: 83692
After: 83692
price_per_kwh      0
total_demand_mw    0
dtype: int64


In [8]:
ausgrid_merge['net_load_kwh'] = ausgrid_merge['load_kwh'] - ausgrid_merge['pv_kwh']
ausgrid_merge.rename(columns = {"Customer": "customer_id"}, inplace=True)
ausgrid_merge.sort_values(by=['customer_id', 'timestamp'], inplace=True)

In [ ]:
final = ausgrid_merge[[
    'customer_id', 'timestamp', 'pv_kwh', 'load_kwh',
    'price_per_kwh', 'total_demand_mw', 'net_load_kwh'
]].copy()
final['timestamp'] = final['timestamp'].dt.tz_convert('Australia/Sydney')
final.to_csv('data/merged_30min_v2.csv', index=False)
print(final.columns.tolist())
print(final.shape)
print((final['net_load_kwh'] < 0).sum())   # tem que ter linhas negativas (exportação solar de dia)
print(final.head())

['customer_id', 'timestamp', 'pv_kwh', 'load_kwh', 'price_per_kwh', 'total_demand_mw', 'net_load_kwh']
(83692, 7)
21849
     customer_id                 timestamp  pv_kwh  load_kwh  price_per_kwh  \
289            1 2012-07-01 00:30:00+10:00     0.0     0.855        0.05704   
310            1 2012-07-01 01:00:00+10:00     0.0     0.786        0.05369   
311            1 2012-07-01 01:30:00+10:00     0.0     0.604        0.05194   
320            1 2012-07-01 02:00:00+10:00     0.0     0.544        0.05213   
321            1 2012-07-01 02:30:00+10:00     0.0     0.597        0.04744   

     total_demand_mw  net_load_kwh  
289          8097.93         0.855  
310          7852.57         0.786  
311          7632.28         0.604  
320          7384.70         0.544  
321          7112.43         0.597  
